# Whisper Accent Robustness — Baseline Evaluation
Compares zero-shot `whisper-small` vs LoRA fine-tuned on L2-ARCTIC.

Covers both **scripted** (test split) and **spontaneous** (full set) speech.
Results are cached to CSV — re-run cells freely without re-running inference.

In [1]:
# ── Config ────────────────────────────────────────────────────────────────────
LORA_PATH       = "models/baseline_loraft"
RESULTS_DIR     = "results"
BATCH_SIZE      = 8
BASELINE_ID     = "openai/whisper-small"

# Set True to skip inference and load cached CSVs
SKIP_ZS_SCRIPTED   = True
SKIP_FT_SCRIPTED   = True
SKIP_ZS_SPONTANEOUS = True
SKIP_FT_SPONTANEOUS = True

In [2]:
# ── Imports ───────────────────────────────────────────────────────────────────
import os
import torch
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
from peft import PeftModel
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import re
from jiwer import wer as jiwer_wer, cer as jiwer_cer

from src.utils.audio_utils import bytes_to_array
from src.utils.load_l2arctic import load_scripted, load_spontaneous, split_dataset

os.makedirs(RESULTS_DIR, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

Device: cuda


In [3]:
# ── Shared helpers ────────────────────────────────────────────────────────────

def add_normalized_columns(df, ref_col="text", pred_col="prediction"):
    """Lowercase + strip punctuation for fair WER comparison."""
    def norm(s):
        if not isinstance(s, str): return ""
        s = s.lower().strip()
        s = re.sub(r"[^\w\s]", "", s)
        s = re.sub(r"\s+", " ", s).strip()
        return s
    df = df.copy()
    df["reference_norm"]  = df[ref_col].apply(norm)
    df["prediction_norm"] = df[pred_col].apply(norm)
    return df


def attach_utterance_stats(df):
    """Add word count and character count columns from reference_norm."""
    df = df.copy()
    df["ref_num_words"] = df["reference_norm"].str.split().str.len()
    df["ref_num_chars"] = df["reference_norm"].str.len()
    return df


def compute_metrics_df(df, ref_col="reference_norm", pred_col="prediction_norm"):
    """Corpus-level WER and CER over a results DataFrame."""
    refs  = df[ref_col].fillna("").tolist()
    preds = df[pred_col].fillna("").tolist()
    return {
        "wer": jiwer_wer(refs, preds),
        "cer": jiwer_cer(refs, preds),
    }


def compute_grouped_metrics(df, group_col="speaker_native_language",
                             ref_col="reference_norm", pred_col="prediction_norm"):
    """Per-group (default: per-L1) WER and CER."""
    rows = []
    for grp, sub in df.groupby(group_col):
        refs  = sub[ref_col].fillna("").tolist()
        preds = sub[pred_col].fillna("").tolist()
        rows.append({
            group_col:    grp,
            "num_utts":   len(sub),
            "wer":        jiwer_wer(refs, preds),
            "cer":        jiwer_cer(refs, preds),
        })
    return pd.DataFrame(rows)


def load_model_zeroshot():
    processor = WhisperProcessor.from_pretrained(BASELINE_ID)
    model = WhisperForConditionalGeneration.from_pretrained(BASELINE_ID).to(DEVICE)
    model.eval()
    model.generation_config.suppress_tokens = None
    model.generation_config.begin_suppress_tokens = None
    return processor, model


def load_model_loraft():
    processor  = WhisperProcessor.from_pretrained(LORA_PATH)
    base       = WhisperForConditionalGeneration.from_pretrained(BASELINE_ID)
    model      = PeftModel.from_pretrained(base, LORA_PATH).to(DEVICE)
    model.eval()
    model.generation_config.suppress_tokens = None
    model.generation_config.begin_suppress_tokens = None
    return processor, model


def transcribe(df, processor, model):
    predictions = []
    for start in range(0, len(df), BATCH_SIZE):
        batch_df = df.iloc[start:start + BATCH_SIZE]
        audio_arrays = [bytes_to_array(row["audio"]["bytes"]) for _, row in batch_df.iterrows()]
        inputs = processor(
            audio_arrays, sampling_rate=16000,
            return_tensors="pt", truncation=True,
            return_attention_mask=True,
        )
        with torch.no_grad():
            pred_ids = model.generate(
                inputs.input_features.to(DEVICE),
                attention_mask=inputs.attention_mask.to(DEVICE),
                language="en", task="transcribe",
            )
        predictions.extend(processor.batch_decode(pred_ids, skip_special_tokens=True))
        if (start // BATCH_SIZE) % 10 == 0:
            print(f"  {start}/{len(df)}", end="\r")
    print(f"  {len(df)}/{len(df)} ✓")
    return predictions


def build_results(df, predictions):
    results = df.drop(columns=["audio"]).copy()
    results["prediction"] = predictions
    results = add_normalized_columns(results, ref_col="text", pred_col="prediction")
    results = attach_utterance_stats(results)
    results["utt_wer"] = results.apply(
        lambda r: jiwer_wer(r["reference_norm"], r["prediction_norm"]) if r["reference_norm"] else None, axis=1
    )
    return results


def run_or_load(skip_flag, csv_path, df, load_model_fn, label):
    """Run inference or load from cache."""
    if skip_flag and os.path.exists(csv_path):
        print(f"Loading cached [{label}] from {csv_path}")
        return pd.read_csv(csv_path)
    print(f"Running inference [{label}]...")
    processor, model = load_model_fn()
    preds   = transcribe(df, processor, model)
    results = build_results(df, preds)
    results.to_csv(csv_path, index=False)
    del model
    torch.cuda.empty_cache()
    print(f"  Saved to {csv_path}")
    return results


def comparison_table(zs_results, ft_results):
    zs_by_l1 = compute_grouped_metrics(zs_results)
    ft_by_l1 = compute_grouped_metrics(ft_results)
    merged = zs_by_l1.rename(columns={"wer": "WER_zeroshot", "cer": "CER_zeroshot"}).merge(
        ft_by_l1.rename(columns={"wer": "WER_loraft", "cer": "CER_loraft"}),
        on=["speaker_native_language", "num_utts"]
    )
    merged["WER_delta"]   = merged["WER_loraft"] - merged["WER_zeroshot"]
    merged["WER_rel_imp"] = -merged["WER_delta"] / merged["WER_zeroshot"] * 100
    return merged.sort_values("WER_zeroshot")


print("Helpers loaded.")

Helpers loaded.


In [4]:
# ── Load datasets ─────────────────────────────────────────────────────────────
print("Loading scripted...")
scripted_df = load_scripted()
_, _, scripted_test = split_dataset(scripted_df)
scripted_test = scripted_test.reset_index(drop=True)

print("Loading spontaneous...")
spontaneous_test = load_spontaneous().reset_index(drop=True)  # Full set (too small to split)

print(f"Scripted test : {len(scripted_test)} utterances")
print(f"Spontaneous   : {len(spontaneous_test)} utterances")

Loading scripted...
Split summary:
  Train :  2294 rows  (63.7%)
  Dev   :   405 rows  (11.3%)
  Test  :   900 rows  (25.0%)  [['BWC', 'EBVS', 'HJK', 'HQTV', 'SKA', 'SVBI']]
Loading spontaneous...
Scripted test : 900 utterances
Spontaneous   : 22 utterances


---
# Part 1 — Scripted Speech

In [5]:
zs_scripted = run_or_load(
    SKIP_ZS_SCRIPTED, f"{RESULTS_DIR}/zeroshot_scripted_predictions.csv",
    scripted_test, load_model_zeroshot, "zero-shot scripted"
)
ft_scripted = run_or_load(
    SKIP_FT_SCRIPTED, f"{RESULTS_DIR}/loraft_scripted_predictions.csv",
    scripted_test, load_model_loraft, "loraft scripted"
)

zs_s_overall = compute_metrics_df(zs_scripted)
ft_s_overall = compute_metrics_df(ft_scripted)
print(f"\nScripted  |  Zero-shot WER: {zs_s_overall['wer']:.4f}  |  LoRA FT WER: {ft_s_overall['wer']:.4f}")

Loading cached [zero-shot scripted] from results/zeroshot_scripted_predictions.csv
Loading cached [loraft scripted] from results/loraft_scripted_predictions.csv

Scripted  |  Zero-shot WER: 0.0992  |  LoRA FT WER: 0.0592


In [6]:
# ── Scripted overall table ────────────────────────────────────────────────────
s_summary = pd.DataFrame([
    {"Model": "Zero-shot", "WER": zs_s_overall["wer"], "CER": zs_s_overall["cer"]},
    {"Model": "LoRA FT",   "WER": ft_s_overall["wer"], "CER": ft_s_overall["cer"]},
])
s_summary["WER Δ"] = s_summary["WER"] - s_summary.loc[0, "WER"]
display(s_summary.style.format({"WER": "{:.4f}", "CER": "{:.4f}", "WER Δ": "{:+.4f}"}).set_caption("Scripted — Overall"))

,Model,WER,CER,WER Δ
0,Zero-shot,0.0992,0.0516,+0.0000
1,LoRA FT,0.0592,0.0299,-0.0400


In [7]:
# ── Scripted per-L1 table ─────────────────────────────────────────────────────
s_l1 = comparison_table(zs_scripted, ft_scripted)
s_l1.to_csv(f"{RESULTS_DIR}/comparison_scripted_by_l1.csv", index=False)
display(s_l1.style.format({
    "WER_zeroshot": "{:.4f}", "CER_zeroshot": "{:.4f}",
    "WER_loraft":   "{:.4f}", "CER_loraft":   "{:.4f}",
    "WER_delta":    "{:+.4f}", "WER_rel_imp":  "{:.1f}%",
}).background_gradient(subset=["WER_delta"], cmap="RdYlGn_r").set_caption("Scripted — By L1"))

,speaker_native_language,num_utts,WER_zeroshot,CER_zeroshot,WER_loraft,CER_loraft,WER_delta,WER_rel_imp
2,Hindi,90,0.0493,0.0243,0.0168,0.0086,-0.0325,65.9%
3,Korean,90,0.0648,0.0361,0.0278,0.0158,-0.0370,57.1%
0,Arabic,90,0.0899,0.0451,0.0579,0.0272,-0.0320,35.6%
1,Chinese,90,0.0968,0.0583,0.0595,0.0311,-0.0373,38.6%
4,Spanish,90,0.1145,0.0554,0.0641,0.0277,-0.0504,44.0%
5,Vietnamese,90,0.1753,0.0881,0.1256,0.0672,-0.0498,28.4%


In [8]:
# ── Scripted charts ───────────────────────────────────────────────────────────
fig = make_subplots(rows=1, cols=2, subplot_titles=("WER by L1", "Relative WER Improvement by L1"))

l1s = s_l1["speaker_native_language"].tolist()
fig.add_trace(go.Bar(name="Zero-shot", x=l1s, y=s_l1["WER_zeroshot"].tolist(), showlegend=True), row=1, col=1)
fig.add_trace(go.Bar(name="LoRA FT",   x=l1s, y=s_l1["WER_loraft"].tolist(),   showlegend=True), row=1, col=1)

colors = ["#e05c5c" if v < 0 else "#5cb85c" for v in s_l1["WER_rel_imp"]]
fig.add_trace(go.Bar(
    x=l1s, y=s_l1["WER_rel_imp"].tolist(),
    marker_color=colors,
    text=[f"{v:+.1f}%" for v in s_l1["WER_rel_imp"]], textposition="outside",
    showlegend=False,
), row=1, col=2)

fig.update_layout(title="Scripted Speech — WER Analysis", barmode="group",
                  legend=dict(orientation="h", yanchor="bottom", y=1.08, xanchor="center", x=0.5))
fig.update_yaxes(title_text="WER", tickformat=".0%", row=1, col=1)
fig.update_yaxes(title_text="Rel. Improvement %", row=1, col=2)
fig.update_xaxes(title_text="L1")
fig.show()

---
# Part 2 — Spontaneous Speech

In [9]:
zs_spontaneous = run_or_load(
    SKIP_ZS_SPONTANEOUS, f"{RESULTS_DIR}/zeroshot_spontaneous_predictions.csv",
    spontaneous_test, load_model_zeroshot, "zero-shot spontaneous"
)
ft_spontaneous = run_or_load(
    SKIP_FT_SPONTANEOUS, f"{RESULTS_DIR}/loraft_spontaneous_predictions.csv",
    spontaneous_test, load_model_loraft, "loraft spontaneous"
)

zs_sp_overall = compute_metrics_df(zs_spontaneous)
ft_sp_overall = compute_metrics_df(ft_spontaneous)
print(f"\nSpontaneous  |  Zero-shot WER: {zs_sp_overall['wer']:.4f}  |  LoRA FT WER: {ft_sp_overall['wer']:.4f}")

Loading cached [zero-shot spontaneous] from results/zeroshot_spontaneous_predictions.csv
Loading cached [loraft spontaneous] from results/loraft_spontaneous_predictions.csv

Spontaneous  |  Zero-shot WER: 0.6349  |  LoRA FT WER: 0.6196


In [10]:
# ── Spontaneous overall table ─────────────────────────────────────────────────
sp_summary = pd.DataFrame([
    {"Model": "Zero-shot", "WER": zs_sp_overall["wer"], "CER": zs_sp_overall["cer"]},
    {"Model": "LoRA FT",   "WER": ft_sp_overall["wer"], "CER": ft_sp_overall["cer"]},
])
sp_summary["WER Δ"] = sp_summary["WER"] - sp_summary.loc[0, "WER"]
display(sp_summary.style.format({"WER": "{:.4f}", "CER": "{:.4f}", "WER Δ": "{:+.4f}"}).set_caption("Spontaneous — Overall"))

,Model,WER,CER,WER Δ
0,Zero-shot,0.6349,0.6133,+0.0000
1,LoRA FT,0.6196,0.6000,-0.0152


In [11]:
# ── Spontaneous per-L1 table ──────────────────────────────────────────────────
sp_l1 = comparison_table(zs_spontaneous, ft_spontaneous)
sp_l1.to_csv(f"{RESULTS_DIR}/comparison_spontaneous_by_l1.csv", index=False)
display(sp_l1.style.format({
    "WER_zeroshot": "{:.4f}", "CER_zeroshot": "{:.4f}",
    "WER_loraft":   "{:.4f}", "CER_loraft":   "{:.4f}",
    "WER_delta":    "{:+.4f}", "WER_rel_imp":  "{:.1f}%",
}).background_gradient(subset=["WER_delta"], cmap="RdYlGn_r").set_caption("Spontaneous — By L1"))

,speaker_native_language,num_utts,WER_zeroshot,CER_zeroshot,WER_loraft,CER_loraft,WER_delta,WER_rel_imp
2,Hindi,3,0.4265,0.4049,0.4382,0.4132,+0.0118,-2.8%
0,Arabic,3,0.5361,0.5261,0.5180,0.5064,-0.0180,3.4%
5,Vietnamese,4,0.5467,0.4812,0.5247,0.4618,-0.0220,4.0%
4,Spanish,4,0.6012,0.5785,0.5877,0.5706,-0.0135,2.2%
1,Chinese,4,0.6418,0.6212,0.6145,0.5933,-0.0273,4.2%
3,Korean,4,0.8028,0.7962,0.7876,0.7846,-0.0152,1.9%


In [12]:
# ── Spontaneous charts ────────────────────────────────────────────────────────
fig = make_subplots(rows=1, cols=2, subplot_titles=("WER by L1", "Relative WER Improvement by L1"))

l1s_sp = sp_l1["speaker_native_language"].tolist()
fig.add_trace(go.Bar(name="Zero-shot", x=l1s_sp, y=sp_l1["WER_zeroshot"].tolist(), showlegend=True), row=1, col=1)
fig.add_trace(go.Bar(name="LoRA FT",   x=l1s_sp, y=sp_l1["WER_loraft"].tolist(),   showlegend=True), row=1, col=1)

colors_sp = ["#e05c5c" if v < 0 else "#5cb85c" for v in sp_l1["WER_rel_imp"]]
fig.add_trace(go.Bar(
    x=l1s_sp, y=sp_l1["WER_rel_imp"].tolist(),
    marker_color=colors_sp,
    text=[f"{v:+.1f}%" for v in sp_l1["WER_rel_imp"]], textposition="outside",
    showlegend=False,
), row=1, col=2)

fig.update_layout(title="Spontaneous Speech — WER Analysis", barmode="group",
                  legend=dict(orientation="h", yanchor="bottom", y=1.08, xanchor="center", x=0.5))
fig.update_yaxes(title_text="WER", tickformat=".0%", row=1, col=1)
fig.update_yaxes(title_text="Rel. Improvement %", row=1, col=2)
fig.update_xaxes(title_text="L1")
fig.show()

---
# Part 3 — Scripted vs Spontaneous Gap

In [13]:
# ── Scripted vs Spontaneous WER gap table ─────────────────────────────────────
gap = pd.DataFrame([
    {"Split": "Scripted",     "Model": "Zero-shot", "WER": zs_s_overall["wer"],  "CER": zs_s_overall["cer"]},
    {"Split": "Scripted",     "Model": "LoRA FT",   "WER": ft_s_overall["wer"],  "CER": ft_s_overall["cer"]},
    {"Split": "Spontaneous",  "Model": "Zero-shot", "WER": zs_sp_overall["wer"], "CER": zs_sp_overall["cer"]},
    {"Split": "Spontaneous",  "Model": "LoRA FT",   "WER": ft_sp_overall["wer"], "CER": ft_sp_overall["cer"]},
])
display(gap.style.format({"WER": "{:.4f}", "CER": "{:.4f}"}).set_caption("Scripted vs Spontaneous"))

,Split,Model,WER,CER
0,Scripted,Zero-shot,0.0992,0.0516
1,Scripted,LoRA FT,0.0592,0.0299
2,Spontaneous,Zero-shot,0.6349,0.6133
3,Spontaneous,LoRA FT,0.6196,0.6000


In [14]:
# ── 4-bar split × model chart ─────────────────────────────────────────────────
fig = go.Figure()
for model_name in ["Zero-shot", "LoRA FT"]:
    sub = gap[gap["Model"] == model_name]
    fig.add_trace(go.Bar(
        name=model_name,
        x=sub["Split"].tolist(),
        y=sub["WER"].tolist(),
        text=[f"{v:.3f}" for v in sub["WER"]], textposition="outside",
    ))

fig.update_layout(
    title="WER: Scripted vs Spontaneous by Model",
    barmode="group",
    legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="center", x=0.5),
)
fig.update_yaxes(title_text="WER", tickformat=".0%")
fig.update_xaxes(title_text="Speech Type")
fig.show()

In [15]:
# ── Per-L1 scripted vs spontaneous WER (zero-shot only) ──────────────────────
zs_s_by_l1  = compute_grouped_metrics(zs_scripted).rename(columns={"wer": "WER_scripted"})
zs_sp_by_l1 = compute_grouped_metrics(zs_spontaneous).rename(columns={"wer": "WER_spontaneous"})
gap_l1 = zs_s_by_l1[["speaker_native_language", "WER_scripted"]].merge(
    zs_sp_by_l1[["speaker_native_language", "WER_spontaneous"]], on="speaker_native_language"
)
gap_l1["gap"] = gap_l1["WER_spontaneous"] - gap_l1["WER_scripted"]

fig = go.Figure()
l1s_gap = gap_l1["speaker_native_language"].tolist()
fig.add_trace(go.Bar(name="Scripted",     x=l1s_gap, y=gap_l1["WER_scripted"].tolist()))
fig.add_trace(go.Bar(name="Spontaneous",  x=l1s_gap, y=gap_l1["WER_spontaneous"].tolist()))
fig.update_layout(
    title="Zero-shot WER by L1 — Scripted vs Spontaneous",
    barmode="group",
    legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="center", x=0.5),
)
fig.update_yaxes(title_text="WER", tickformat=".0%")
fig.update_xaxes(title_text="L1 Background")
fig.show()

---
# Part 4 — Qualitative Examples

In [16]:
# ── Worst scripted zero-shot utterances + FT comparison ───────────────────────
worst_s = zs_scripted.nlargest(10, "utt_wer")[["text", "speaker_native_language", "reference_norm", "prediction_norm", "utt_wer"]]
worst_s = worst_s.rename(columns={"prediction_norm": "zs_pred", "utt_wer": "zs_wer"})
worst_s["ft_pred"] = ft_scripted.loc[worst_s.index, "prediction_norm"].values
worst_s["ft_wer"]  = ft_scripted.loc[worst_s.index, "utt_wer"].values
worst_s["delta"]   = worst_s["zs_wer"] - worst_s["ft_wer"]
display(worst_s.sort_values("delta", ascending=False)
        .style.format({"zs_wer": "{:.2f}", "ft_wer": "{:.2f}", "delta": "{:+.2f}"})
        .set_caption("Top-10 Worst Scripted (Zero-shot) — with FT comparison"))

,text,speaker_native_language,reference_norm,zs_pred,zs_wer,ft_pred,ft_wer,delta
102,it is the fire partly she said,Vietnamese,it is the fire partly she said,is it the 5 partly cz,0.71,it is the fire partly she said,0.00,+0.71
203,death had come with terrible suddenness,Korean,death had come with terrible suddenness,that sad comedy tad of a suddenness,1.00,thats had come with the terrible suddenness,0.33,+0.67
37,the gray eyes faltered the flush deepened,Spanish,the gray eyes faltered the flush deepened,the gray light falter it the flush deepen it,0.71,the gray light faltered the flush deepened,0.14,+0.57
89,outsiders are allowed five minute speeches the sick man urged,Vietnamese,outsiders are allowed five minute speeches the sick man urged,outside there are low 5 minutes beaches the sick mange urge,0.80,outside the low five minute beaches the sick man urged,0.40,+0.40
417,it is the fire partly she said,Chinese,it is the fire partly she said,it is the file partially reset,0.57,it is the file partially she said,0.29,+0.29
482,in partnership with daylight the pair raided the san jose interurban,Spanish,in partnership with daylight the pair raided the san jose interurban,in partnerships with daylight the payer rated the san jose in the urban,0.55,in partnerships with daylight the pair raided the sanjos in teruban,0.36,+0.18
9,providence had delivered him through the maelstrom,Vietnamese,providence had delivered him through the maelstrom,provident has delivered him through the myth charm,0.57,provident has delivered him through the mistrum,0.43,+0.14
35,outsiders are allowed five minute speeches the sick man urged,Vietnamese,outsiders are allowed five minute speeches the sick man urged,outsider has loud feminist speech the sixth manner,0.90,outside us love feminists preach the six men work,0.90,+0.00
167,i had faith in them,Spanish,i had faith in them,that i've fade in them,0.60,but i've failed in them,0.60,+0.00
275,only it is so wonderful so almost impossible to believe,Vietnamese,only it is so wonderful so almost impossible to believe,only is the show wonderful and so most impossible to believe,0.50,only it is the show only fool so most impossible to believe,0.50,+0.00


In [17]:
# ── Worst spontaneous utterances ──────────────────────────────────────────────
worst_sp = zs_spontaneous.nlargest(10, "utt_wer")[["text", "speaker_native_language", "reference_norm", "prediction_norm", "utt_wer"]]
worst_sp = worst_sp.rename(columns={"prediction_norm": "zs_pred", "utt_wer": "zs_wer"})
worst_sp["ft_pred"] = ft_spontaneous.loc[worst_sp.index, "prediction_norm"].values
worst_sp["ft_wer"]  = ft_spontaneous.loc[worst_sp.index, "utt_wer"].values
worst_sp["delta"]   = worst_sp["zs_wer"] - worst_sp["ft_wer"]
display(worst_sp.sort_values("delta", ascending=False)
        .style.format({"zs_wer": "{:.2f}", "ft_wer": "{:.2f}", "delta": "{:+.2f}"})
        .set_caption("Top-10 Worst Spontaneous (Zero-shot) — with FT comparison"))

,text,speaker_native_language,reference_norm,zs_pred,zs_wer,ft_pred,ft_wer,delta
17,uh it's this story happened in a big city i think it's a rather tall building or an apartment building and there is a woman and a man each has a suitcase in their hands and uh when they turn the corner they uh run into each other and the suitcases were on the ground and so they when they rise again and they picked up the suitcase and went their sep separate ways but when they get to their i guess their apartments they found out that they took the wrong suitcase the man now had the woman's suitcase and he's smiling weirdly at those uh clothings and the woman was surprised to find that there was a tie in the suitcase and she must have realized that she took the wrong suitcase,Chinese,uh it's this story happened in a big city i think it's a rather tall building or an apartment building and there is a woman and a man each has a suitcase in their hands and uh when they turn the corner they uh run into each other and the suitcases were on the ground and so they when they rise again and they picked up the suitcase and went their sep separate ways but when they get to their i guess their apartments they found out that they took the wrong suitcase the man now had the woman's suitcase and he's smiling weirdly at those uh clothings and the woman was surprised to find that there was a tie in the suitcase and she must have realized that she took the wrong suitcase,this story happened in a big city i think it's a rather tall building an apartment building there are women and men each has a suitcase in their hands when they turn the corner they run into each other and the suitcases were on the ground when they rise again,0.65,it's this story happened in a big city i think it's a rather tall building or an apartment building and there's a women and men each has a suitcase in their hands and when they turn the corner they run into each other and the suitcases were on the ground and so they when they rise again and they,0.58,+0.07
5,um okay in the city i think um there are many skyscrapers and um the many cars uh not so many cars on the road and in the corner of the building the two people are walking into the same directions and they're carrying the same carrier i think i mean the suitcase i think um because they did not see each other they just bumped into each other and um and especially for the man he dropped his glasses um after bumping into each other um they seem to have a headache because there is the uh stars going around their over their heads and after that they said to just the i'm sorry to each other and um pick up their own suitcases but and they just did uh went to their own direction i mean their destinations but after that when just they opened their um suitcases they found out it it's just they pick up the wrong suitcase because for the man in his suitcase they i mean he just he found out that uh the red underwear was in the suitcase whereas the the woman found out that in her suitcase there was the um the yellow striped uh necktie in the suitcase so,Korean,um okay in the city i think um there are many skyscrapers and um the many cars uh not so many cars on the road and in the corner of the building the two people are walking into the same directions and they're carrying the same carrier i think i mean the suitcase i think um because they did not see each other they just bumped into each other and um and especially for the man he dropped his glasses um after bumping into each other um they seem to have a headache because there is the uh stars going around their over their heads and after that they said to just the i'm sorry to each other and um pick up their own suitcases but and they just did uh went to their own direction i mean their destinations but after that when just they opened their um suitcases they found out it it's just they pick up the wrong suitcase because for the man in his suitcase they i mean he just he found out that uh the r

In [18]:
# ── Utterance WER distribution: scripted vs spontaneous ───────────────────────
fig = go.Figure()
fig.add_trace(go.Histogram(x=zs_scripted["utt_wer"],     name="Scripted ZS",  opacity=0.6, nbinsx=30))
fig.add_trace(go.Histogram(x=zs_spontaneous["utt_wer"],  name="Spontaneous ZS", opacity=0.6, nbinsx=30))
fig.add_trace(go.Histogram(x=ft_scripted["utt_wer"],     name="Scripted FT",  opacity=0.6, nbinsx=30))
fig.add_trace(go.Histogram(x=ft_spontaneous["utt_wer"],  name="Spontaneous FT", opacity=0.6, nbinsx=30))
fig.update_layout(
    title="Utterance WER Distribution — All Conditions",
    barmode="overlay",
    legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="center", x=0.5),
)
fig.update_xaxes(title_text="Utterance WER")
fig.update_yaxes(title_text="Count")
fig.show()